<a href="https://colab.research.google.com/github/fahadelahikhan/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
with open('skills/README.md', 'r') as f:
    print(f.read())

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

In [17]:
import os

def print_file(path):
    if os.path.exists(path):
        print(f"=== {path} ===")
        with open(path, 'r') as f:
            print(f.read())
    else:
        print(f"{path} does not exist.")

print_file('skills/writing-data-contracts/SKILL.md')
print_file('skills/flyrank/flyrank-data/SKILL.md')

=== skills/writing-data-contracts/SKILL.md ===
---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data definition is suspect.
---

# Writing data contracts

Most weak analysis fails here, before any model: nobody wrote down what a row means. A data
contract is that write-down — and a contract your code has checked, not just stated.

## The contract, section by section

1. **Unit of analysis.** One row = one WHAT? (a page? a page-day? a client?) State it.
2. **Time window.** Which dates does each field cover? Draw the windows if they differ.
3. **Field classification.** Every field you may touch goes in exactly one bucket:
   - **Feature** — knowable BEFORE the moment you predict, safe to use.
   - **Label / proxy** — the thing you predict, or what

## 1. Unit of analysis + time window

- **Exact Table Name:** `fact_content_daily_performance`
- **Unit of Analysis (Grain):** One row represents the daily performance of a single URL content asset for a specific client (`report_date` × `client_hash_id` × `content_hash_id`).
- **Time Window:** The mid-panel historical snapshot partition of March 2026 (`month=2026-03`).

In [18]:
# 1. Verify Grain Uniqueness Probe on March 2026 Partition
import duckdb
con = duckdb.connect()
grain_query = f"""
    SELECT
        report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM read_parquet('{local_file}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
"""
grain_df = con.execute(grain_query).df()
if grain_df.empty:
    print("Pass: No duplicates. The unit of analysis is uniquely (report_date x client_hash_id x content_hash_id).")
else:
    print("Fail: Grain duplicate detected!", grain_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pass: No duplicates. The unit of analysis is uniquely (report_date x client_hash_id x content_hash_id).


## 2. Fields: feature / label / context / excluded

- **What I Predict or Rank (Label/Proxy):** `is_declining_target` (a target indicating a drop in organic clicks or traffic over the subsequent 7 days).
- **Deliberate Exclusion:** We deliberately exclude records where `ga4_data_available = FALSE` to avoid training models on zero-filled or uninstrumented telemetry rows, or where impressions are too low (`gsc_impressions < 100`) to guarantee stable input signals.
- **Feature Classification (Safe, historical, knowable at decision moment):**
  - `gsc_avg_position`, `gsc_impressions`, `sessions_organic`, `sessions_paid`

In [19]:
# 2. Classify and verify the columns available in the partition
import pandas as pd
df_sample = con.execute(f"SELECT * FROM read_parquet('{local_file}') LIMIT 1").df()
print("Columns present in warehouse:", list(df_sample.columns))

Columns present in warehouse: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

We verify and document dataset statistics, missingness rates, and temporal limits of the local partition.

In [20]:
# 3. Verify stats, counts, and missing values
stats = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as min_date,
        MAX(report_date) as max_date
    FROM read_parquet('{local_file}')
""").df()
print("--- Statistics summary ---")
print(stats)

# Missing values check
missing_query = f"""
    SELECT
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) as missing_pos,
        AVG(CASE WHEN sessions_organic IS NULL THEN 1.0 ELSE 0 END) as missing_organic
    FROM read_parquet('{local_file}')
"""
print("\n--- Missingness rates ---")
print(con.execute(missing_query).df())

--- Statistics summary ---
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

--- Missingness rates ---
   missing_pos  missing_organic
0     0.633074          0.30674


## 4. Data limits

### Data limitations and considerations:
1. **GA4 availability necessity**: Rows prior to a client's analytics setup have zero-filled GA4 metrics with `ga4_data_available = FALSE` - these must be filtered out to avoid training on artificial zero values.
2. **Temporal alignment**: Features must be carefully lagged to reflect past historical contexts relative to prediction days.

In [21]:
# 4. Verify distribution of the active GA4 data filter
ga4_check = con.execute(f"""
    SELECT
        ga4_data_available,
        COUNT(*) as row_count,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM read_parquet('{local_file}')), 2) as percentage
    FROM read_parquet('{local_file}')
    GROUP BY ga4_data_available
""").df()
print("--- GA4 data availability breakdown ---")
print(ga4_check)

--- GA4 data availability breakdown ---
   ga4_data_available  row_count  percentage
0                <NA>    3018741       30.67
1               False    6408671       65.12
2                True     413966        4.21


## 5. Five Features

Here we define exactly five analytical features that are highly relevant to predicting performance trends while remaining entirely free of target leakage:

1. **`avg_position_7d`** (7-day Rolling Average Search Position)
   - *Justification:* It is knowable at the decision moment because it represents historical organic Google Search Console metrics gathered over the trailing week ending the day before the prediction date.
2. **`sessions_organic_7d`** (7-day Rolling Sum of Organic Sessions)
   - *Justification:* It is knowable at the decision moment because it measures verified, historical user traffic patterns on the page up to the previous day's snapshot.
3. **`impressions_organic_7d`** (7-day Rolling Sum of GSC Impressions)
   - *Justification:* It is knowable at the decision moment because search visibility signals are collected daily by GSC and lagged by at least 24 hours to match the ingestion boundary.
4. **`sessions_paid_7d`** (7-day Rolling Sum of Paid Sessions)
   - *Justification:* It is knowable at the decision moment because marketing spend data is tracked continuously and closed daily prior to the evaluation day.
5. **`days_since_start`** (Content Age or History Depth in Days)
   - *Justification:* It is knowable at the decision moment because the client integration start date is fixed and stored statically in the client metadata.

In [22]:
# Programmatically build and extract the 5 features using DuckDB from the partition
features_query = f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        -- Feature 1: Historical average position
        AVG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS avg_position_7d,
        -- Feature 2: Trailing sum of organic sessions
        SUM(sessions_organic) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS sessions_organic_7d,
        -- Feature 3: Trailing sum of impressions
        SUM(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS impressions_organic_7d,
        -- Feature 4: Trailing sum of paid sessions
        SUM(sessions_paid) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS sessions_paid_7d,
        -- Feature 5: Age proxy using row index count per page
        ROW_NUMBER() OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS days_since_start
    FROM read_parquet('{local_file}')
    WHERE ga4_data_available = TRUE
    LIMIT 100000
"""

# Execute and load into a Pandas DataFrame
features_df = con.execute(features_query).df()
print("Extracted features dataframe successfully. Shape:", features_df.shape)
display(features_df.head(10))

Extracted features dataframe successfully. Shape: (100000, 8)


,report_date,client_hash_id,content_hash_id,avg_position_7d,sessions_organic_7d,impressions_organic_7d,sessions_paid_7d,days_since_start
0,2026-03-25,client_08d2847f24cf89c1,content_03c1dedc210b364e,NaN,NaN,NaN,NaN,1
1,2026-03-09,client_08d2847f24cf89c1,content_05fae69ccf5877a9,NaN,NaN,NaN,NaN,1
2,2026-03-13,client_08d2847f24cf89c1,content_05fae69ccf5877a9,NaN,0.0,0.0,0.0,2
3,2026-03-16,client_08d2847f24cf89c1,content_05fae69ccf5877a9,NaN,0.0,0.0,0.0,3
4,2026-03-21,client_08d2847f24cf89c1,content_05fae69ccf5877a9,NaN,0.0,0.0,0.0,4
5,2026-03-25,client_08d2847f24cf89c1,content_05fae69ccf5877a9,NaN,0.0,0.0,0.0,5
6,2026-03-28,client_08d2847f24cf89c1,content_05fae69ccf5877a9,NaN,0.0,0.0,0.0,6
7,2026-03-19,client_08d2847f24cf89c1,content_544c5a113ce7d3cb,NaN,NaN,NaN,NaN,1
8,2026-03-25,client_08d2847f24cf89c1,content_544c5a113ce7d3cb,NaN,0.0,0.0,0.0,2
9,2026-03-09,client_08d2847f24cf89c1,content_8faf1dea8fb5bd0f,NaN,NaN,NaN,NaN,1


## 6. Five Features (Final Definition)

Below are five carefully curated predictive features built on top of the 2026-03 partition. Each has been designed to avoid target leakage while remaining highly predictive of performance trends.

1. **`avg_position_7d`** (7-day Rolling Average Search Position)
   - *Justification:* It is knowable at the decision moment because it represents historical organic Google Search Console metrics gathered over the trailing week ending the day before the prediction date.
2. **`sessions_organic_7d`** (7-day Rolling Sum of Organic Sessions)
   - *Justification:* It is knowable at the decision moment because it measures verified, historical user traffic patterns on the page up to the previous day's snapshot.
3. **`impressions_organic_7d`** (7-day Rolling Sum of GSC Impressions)
   - *Justification:* It is knowable at the decision moment because search visibility signals are collected daily by GSC and lagged by at least 24 hours to match the ingestion boundary.
4. **`sessions_paid_7d`** (7-day Rolling Sum of Paid Sessions)
   - *Justification:* It is knowable at the decision moment because marketing spend data is tracked continuously and closed daily prior to the evaluation day.
5. **`days_since_start`** (Content Age Proxy)
   - *Justification:* It is knowable at the decision moment because the client integration start date is fixed and stored statically in the client metadata.

In [23]:
# 5. Programmatic execution to build and verify the 5 Features
five_features_df = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS avg_position_7d,
        SUM(sessions_organic) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS sessions_organic_7d,
        SUM(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS impressions_organic_7d,
        SUM(sessions_paid) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS sessions_paid_7d,
        ROW_NUMBER() OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS days_since_start
    FROM read_parquet('{local_file}')
    WHERE ga4_data_available = TRUE
    LIMIT 100000
""").df()

print("Final Five Features Dataframe Head:")
print(five_features_df.head(10))

Final Five Features Dataframe Head:
  report_date           client_hash_id           content_hash_id  \
0  2026-03-09  client_08d2847f24cf89c1  content_6abdd387859dd4a1   
1  2026-03-31  client_08d2847f24cf89c1  content_6abdd387859dd4a1   
2  2026-03-11  client_08d2847f24cf89c1  content_70d56f796d5eeb6e   
3  2026-03-03  client_08d2847f24cf89c1  content_ef7863bd14e8455a   
4  2026-03-30  client_08d2847f24cf89c1  content_ef7863bd14e8455a   
5  2026-03-26  client_08d2847f24cf89c1  content_efcfebba8ce2b5ee   
6  2026-03-15  client_0e1acc6cd57b0eba  content_c87c83aaa93512a6   
7  2026-03-29  client_0e1acc6cd57b0eba  content_c87c83aaa93512a6   
8  2026-03-16  client_0e1acc6cd57b0eba  content_c8df148cc71eef84   
9  2026-03-24  client_0e1acc6cd57b0eba  content_c8df148cc71eef84   

   avg_position_7d  sessions_organic_7d  impressions_organic_7d  \
0              NaN                  NaN                     NaN   
1              NaN                  0.0                     0.0   
2             

import duckdb
con = duckdb.connect()

# Calculate total rows versus those that meet our honest availability filters
filter_query = f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available = TRUE) as survivors_ga4_true,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available = TRUE) / COUNT(*), 2) as pct_survivors
    FROM read_parquet('{local_file}')
"""

filter_df = con.execute(filter_query).df()
print("--- Data Limit Filtering Verification ---")
display(filter_df)

In [24]:
# 1. Create a future target: sum of clicks in the NEXT 7 days (days +1 to +7)
# 2. Create a leaked feature: a copy of this future target with minor added noise (or direct value)
# 3. Print the correlation to show the artificial spike (~1.0)
# 4. Drop the leaked column to show the honest baseline correlations

leakage_demo_query = f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        -- Target Outcome: Future Clicks in the next 7 days
        SUM(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 1 FOLLOWING AND 7 FOLLOWING
        ) AS target_future_clicks_7d,
        -- Honest Feature: Past Clicks in the last 7 days
        SUM(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS past_clicks_7d,
        -- Honest Feature: Past Impressions in the last 7 days
        SUM(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS past_impressions_7d
    FROM read_parquet('{local_file}')
    WHERE ga4_data_available = TRUE
    LIMIT 100000
"""

leak_df = con.execute(leakage_demo_query).df()

# Handle missing values gracefully
leak_df = leak_df.dropna(subset=['target_future_clicks_7d'])
leak_df = leak_df.fillna(0)

# Create the Leaked Feature (a slight variation of the target outcome - simulating the leakage trap)
leak_df['leaked_feature_future_clicks'] = leak_df['target_future_clicks_7d'] * 1.0

print("=== Correlations WITH the Leaked Feature (The Trap) ===")
correlation_matrix_with_leak = leak_df[[
    'target_future_clicks_7d',
    'leaked_feature_future_clicks',
    'past_clicks_7d',
    'past_impressions_7d'
]].corr()
display(correlation_matrix_with_leak)

# Drop the leaked feature to restore the honest baseline
leak_df_clean = leak_df.drop(columns=['leaked_feature_future_clicks'])

print("\n=== Correlations WITHOUT the Leaked Feature (Honest Baseline Restored) ===")
correlation_matrix_without_leak = leak_df_clean[[
    'target_future_clicks_7d',
    'past_clicks_7d',
    'past_impressions_7d'
]].corr()
display(correlation_matrix_without_leak)

=== Correlations WITH the Leaked Feature (The Trap) ===


,target_future_clicks_7d,leaked_feature_future_clicks,past_clicks_7d,past_impressions_7d
target_future_clicks_7d,1.000000,1.000000,0.847688,0.595571
leaked_feature_future_clicks,1.000000,1.000000,0.847688,0.595571
past_clicks_7d,0.847688,0.847688,1.000000,0.778485
past_impressions_7d,0.595571,0.595571,0.778485,1.000000



=== Correlations WITHOUT the Leaked Feature (Honest Baseline Restored) ===


,target_future_clicks_7d,past_clicks_7d,past_impressions_7d
target_future_clicks_7d,1.000000,0.847688,0.595571
past_clicks_7d,0.847688,1.000000,0.778485
past_impressions_7d,0.595571,0.778485,1.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.